In [1]:
import tvm
import tvm.testing
from tvm.relay import testing
from tvm import relax, relay
from tvm.relax.testing import relay_translator, nn
from tvm.runtime import vm as vm_rt
from tvm.script import relax as R
import numpy as np

[13:27:16] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[13:27:16] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[13:27:16] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[13:27:16] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[13:27:16] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[13:27:16] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled w

In [2]:
relay_mod, _ = testing.resnet.get_workload(num_layers=50, batch_size=1, dtype="float32")

# translate the ResNet model from Relay to Relax
target = tvm.target.Target("llvm", host="llvm")
relax_mod = relay_translator.from_relay(relay_mod["main"], target)

# print the ResNet IRmodule got translated
# relax_mod.show()

# build the IRModule and create relax vm
# ex = relax.build(relax_mod, target)
ex = relax.build(relax_mod, target, pipeline="micro_build", exec_mode="crt")
# vm = relax.VirtualMachine(ex, tvm.cpu())

# init weights and run the model on relax vm
# shape = (1, 3, 224, 224)
# data = tvm.nd.array(np.random.rand(*shape).astype(np.float32))
# params = nn.init_params(relax_mod)
# res = vm["main"](data, *params)

# check correctness by comparing with relay result
# exe = relay.vm.compile(relay_mod, target)
# relay_vm = vm_rt.VirtualMachine(exe, tvm.cpu())
# inputs = [data] + params
# expected_output = relay_vm.run(*inputs)
# tvm.testing.assert_allclose(res.numpy(), expected_output.numpy(), rtol=1e-4, atol=1e-4)

In [3]:
ex

In [68]:
@tvm.instrument.pass_instrument
class MyInstrument:

    def __init__(self):
        self.skip_pass_name = []
        self.output = []
        self.output_after = []
        self.idx = 0

    def run_before_pass(self, mod, pass_info):
        print("run_before_pass", self.idx, pass_info.name)
        print(dir(mod))
        self.idx += 1
        # tmp = (mod.astext(show_meta_data=True), str(pass_info))
        # tmp = (str(mod), str(pass_info))
        tmp = (mod.script(show_meta=True), str(pass_info))
        self.output.append(tmp)


    def run_after_pass(self, mod, pass_info):
        print("run_after_pass", self.idx, pass_info.name)
        print(dir(mod))
        # tmp = (mod.astext(show_meta_data=True), str(pass_info))
        tmp = (str(mod), str(pass_info))
        self.output_after.append(tmp)
        pass


In [9]:
# RELAX
my_instrument = MyInstrument()
with tvm.transform.PassContext(instruments=[my_instrument]):
    ex = relax.build(relax_mod, target, pipeline="micro_build", exec_mode="compiled")

run_before_pass _pipeline
['__class__', '__contains__', '__del__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_handle_by_constructor__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__setstate__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_add', '_import', '_move', '_relax_script', 'astext', 'attrs', 'clone', 'from_expr', 'functions', 'functions_items', 'get_attr', 'get_constructor', 'get_global_type_var', 'get_global_type_vars', 'get_global_var', 'get_global_vars', 'get_type', 'global_infos', 'global_type_var_map_', 'global_var_map_', 'handle', 'import_from_std', 'legacy_repr', 'same_as', 'script', 'show', 'source_map', 'type_definitions', 'update', 'update_func', 'update_global_info', 'with

[13:07:05] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:05] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:05] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:05] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListMoveFromPackedReturn
[13:07:05] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:05] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:05] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: 

run_before_pass tir.LowerCustomDatatypes
['__class__', '__contains__', '__del__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_handle_by_constructor__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__setstate__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_add', '_import', '_move', '_relax_script', 'astext', 'attrs', 'clone', 'from_expr', 'functions', 'functions_items', 'get_attr', 'get_constructor', 'get_global_type_var', 'get_global_type_vars', 'get_global_var', 'get_global_vars', 'get_type', 'global_infos', 'global_type_var_map_', 'global_var_map_', 'handle', 'import_from_std', 'legacy_repr', 'same_as', 'script', 'show', 'source_map', 'type_definitions', 'update', 'update_func', 'update_glob

[13:07:06] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:06] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:06] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:06] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListMoveFromPackedReturn
[13:07:06] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:06] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:06] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: 

run_before_pass tir.LowerIntrin
['__class__', '__contains__', '__del__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_handle_by_constructor__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__setstate__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_add', '_import', '_move', '_relax_script', 'astext', 'attrs', 'clone', 'from_expr', 'functions', 'functions_items', 'get_attr', 'get_constructor', 'get_global_type_var', 'get_global_type_vars', 'get_global_var', 'get_global_vars', 'get_type', 'global_infos', 'global_type_var_map_', 'global_var_map_', 'handle', 'import_from_std', 'legacy_repr', 'same_as', 'script', 'show', 'source_map', 'type_definitions', 'update', 'update_func', 'update_global_info',

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



run_before_pass tir.CombineContextCall
['__class__', '__contains__', '__del__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_handle_by_constructor__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__setstate__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_add', '_import', '_move', '_relax_script', 'astext', 'attrs', 'clone', 'from_expr', 'functions', 'functions_items', 'get_attr', 'get_constructor', 'get_global_type_var', 'get_global_type_vars', 'get_global_var', 'get_global_vars', 'get_type', 'global_infos', 'global_type_var_map_', 'global_var_map_', 'handle', 'import_from_std', 'legacy_repr', 'same_as', 'script', 'show', 'source_map', 'type_definitions', 'update', 'update_func', 'update_global

[13:07:09] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:09] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:09] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:09] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListMoveFromPackedReturn
[13:07:09] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:09] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: Warning: No TScriptPrinterName attribute for tir.TVMBackendAnyListSetPackedArg
[13:07:09] /var/tmp/ga87puy/tvm_relax/src/script/printer/tir/expr.cc:246: 

run_before_pass sequential
['__class__', '__contains__', '__del__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_handle_by_constructor__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__setstate__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_add', '_import', '_move', '_relax_script', 'astext', 'attrs', 'clone', 'from_expr', 'functions', 'functions_items', 'get_attr', 'get_constructor', 'get_global_type_var', 'get_global_type_vars', 'get_global_var', 'get_global_vars', 'get_type', 'global_infos', 'global_type_var_map_', 'global_var_map_', 'handle', 'import_from_std', 'legacy_repr', 'same_as', 'script', 'show', 'source_map', 'type_definitions', 'update', 'update_func', 'update_global_info', 'wit

In [69]:
# NON-RELAX
my_instrument = MyInstrument()
with tvm.transform.PassContext(instruments=[my_instrument], config={"tir.usmp.enable": False}):
    ex = relay.build(conv2d_mod, target=tvm.target.Target("c", host="c"), runtime=tvm.relay.backend.Runtime("crt"), executor=tvm.relay.backend.Executor("aot", {"interface-api": "packed", "unpacked-api": False}))

run_before_pass 0 sequential
['__class__', '__contains__', '__del__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_handle_by_constructor__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__setstate__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_add', '_import', '_move', '_relax_script', 'astext', 'attrs', 'clone', 'from_expr', 'functions', 'functions_items', 'get_attr', 'get_constructor', 'get_global_type_var', 'get_global_type_vars', 'get_global_var', 'get_global_vars', 'get_type', 'global_infos', 'global_type_var_map_', 'global_var_map_', 'handle', 'import_from_std', 'legacy_repr', 'same_as', 'script', 'show', 'source_map', 'type_definitions', 'update', 'update_func', 'update_global_info', 'w

In [55]:
ex

In [64]:
print(my_instrument.output[-6][1])

The meta data of the pass - pass name: tir.BindTarget, opt_level: 0, required passes: []



In [57]:
dir(ex)
print(ex.module._collect_dso_modules()[2].get_source())

// tvm target: c -keys=cpu 
#define TVM_EXPORTS
#include "tvm/runtime/c_runtime_api.h"
#include "tvm/runtime/c_backend_api.h"
#include <math.h>
#include <stdbool.h>

#ifdef __cplusplus
extern "C" {
#endif
static const float __attribute__((section(".rodata.tvm"), aligned(16))) fused_constant[32] = {
    0x1.ec7ddap-1, 0x1.89792cp-1, 0x1.5c5c68p-1, 0x1.30b11ap-4, 0x1.d72d2p-1, 0x1.5afce8p-2, 0x1.784c34p-2, 0x1.ed6314p-2, 
    0x1.669f1ep-1, 0x1.882fd2p-1, 0x1.20ac12p-8, 0x1.abbc44p-1, 0x1.d5c1ep-2, 0x1.9f070ep-2, 0x1.3cd0d8p-1, 0x1.a6a334p-3, 
    0x1.316708p-1, 0x1.c91282p-1, 0x1.ad1584p-1, 0x1.ebe53cp-1, 0x1.d3a44ap-1, 0x1.b3f51cp-1, 0x1.c827dp-1, 0x1.db138cp-1, 
    0x1.75b482p-2, 0x1.6d395cp-5, 0x1.a296a6p-1, 0x1.d604bcp-1, 0x1.96adbp-1, 0x1.4f5eb4p-2, 0x1.413bap-1, 0x1.f1e27ap-1
};
#ifdef __cplusplus
}  // extern "C"
#endif
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t tvmgen_default_fused_nn_conv2d(void* args, int32_t* arg_type_ids, int32_t num_args, void* out_ret_value, int

In [50]:
[(i, my_instrument.output[i][1]) for i in range(len(my_instrument.output))]

[(0,
  'The meta data of the pass - pass name: sequential, opt_level: 0, required passes: []\n'),
 (1,
  'The meta data of the pass - pass name: RemoveUnusedFunctions, opt_level: 1, required passes: []\n'),
 (2,
  'The meta data of the pass - pass name: ToBasicBlockNormalForm, opt_level: 1, required passes: []\n'),
 (3,
  'The meta data of the pass - pass name: qnn.Legalize, opt_level: 0, required passes: []\n'),
 (4,
  'The meta data of the pass - pass name: InferType, opt_level: 0, required passes: []\n'),
 (5,
  'The meta data of the pass - pass name: QnnLegalize, opt_level: 1, required passes: [\nInferType, ]\n'),
 (6,
  'The meta data of the pass - pass name: InferType, opt_level: 0, required passes: []\n'),
 (7,
  'The meta data of the pass - pass name: InferType, opt_level: 0, required passes: []\n'),
 (8,
  'The meta data of the pass - pass name: QnnCanonicalize, opt_level: 1, required passes: [\nInferType, ]\n'),
 (9,
  'The meta data of the pass - pass name: InferType, opt_le

In [51]:
with tvm.transform.PassContext(instruments=[my_instrument]):
    ex = relax.build(relax_mod, target, exec_mode="compiled")

TVMError: Traceback (most recent call last):
  8: tvm::runtime::PackedFuncObj::Extractor<tvm::runtime::PackedFuncSubObj<tvm::runtime::TypedPackedFunc<tvm::IRModule (tvm::relax::ExecBuilder, tvm::IRModule)>::AssignTypedLambda<tvm::IRModule (*)(tvm::relax::ExecBuilder, tvm::IRModule)>(tvm::IRModule (*)(tvm::relax::ExecBuilder, tvm::IRModule), std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >)::{lambda(tvm::runtime::TVMArgs const&, tvm::runtime::TVMRetValue*)#1}> >::Call(tvm::runtime::PackedFuncObj const*, tvm::runtime::TVMArgs, tvm::runtime::TVMRetValue*)
  7: tvm::relax::relax_vm::VMTIRCodeGen(tvm::relax::ExecBuilder, tvm::IRModule)
  6: tvm::relax::relax_vm::CodeGenVMTIR::Run(tvm::relax::ExecBuilder, tvm::IRModule)
  5: tvm::relax::relax_vm::CodeGenVMTIR::Codegen(tvm::relax::Function const&)
  4: tvm::relax::ExprFunctor<tvm::runtime::Optional<tvm::PrimExpr> (tvm::RelayExpr const&)>::VisitExpr(tvm::RelayExpr const&)
  3: _ZZN3tvm5relax11ExprFunctorIFNS_7run

In [55]:
ex.as_text()

'@main num_inputs=260 vm_tir_func;\n\n'

In [56]:
ex.as_python()

'ib = rx.Builder()\n'

In [71]:
ex.mod.save()

TypeError: save() missing 1 required positional argument: 'file_name'

In [79]:
print(my_instrument.output[-7][0])

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    I.module_attrs({"runtime": None})
    @T.prim_func
    def __vmtir__main(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""})})
        assert num_args == 4, "__vmtir__main: num_args should be 4"
        assert not T.isnullptr(args), "__vmtir__main: TVMValue* arg pointer was NULL"
        assert not T.isnullptr(arg_type_ids), "__vmtir__main: int* type_codes was NULL"
        arg_type_ids_1 = T.decl_buffer((4,), "int32", data=arg_type_ids)
        ctx_ptr_code: T.int32 = arg_type_ids_1[0]
        assert ctx_ptr_code == 3 or ctx_ptr_code == 13 or ctx_ptr_code == 7 or ctx_ptr_code == 4, "__vmtir__main: Expect arg[0] to be pointer"


In [27]:
def relay_conv2d():
    """
    Simple conv2d Relay implementation.
    """
    dtype = "float32"

    x = relay.var("x", shape=(1, 4, 2, 2), dtype=dtype)
    weight = relay.const(np.random.uniform(size=(2, 4, 2, 2)), dtype=dtype)
    x = relay.nn.softmax(relay.nn.conv2d(x, weight, kernel_size=(2, 2),))
    # x = relay.nn.conv2d(x, weight, kernel_size=(2, 2),)
    func = relay.Function(relay.analysis.free_vars(x), x)
    # return tvm.IRModule.from_expr(func)
    return func
conv2d_mod = relay_conv2d()

In [91]:
conv2d_mod

fn (%x: Tensor[(1, 4, 2, 2), float32]) {
  nn.conv2d(%x, meta[relay.Constant][0], padding=[0, 0, 0, 0], kernel_size=[2, 2])
}

In [92]:
relax_mod = relay_translator.from_relay(conv2d_mod, target)

In [93]:
relax_mod

# from tvm.script import ir as I
# from tvm.script import tir as T
# from tvm.script import relax as R

@I.ir_module
class Module:
    @T.prim_func(private=True)
    def contrib_conv2d_NCHWc(A: T.Buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), "float32"), B: T.Buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), "float32"), conv2d_NCHWc: T.Buffer((T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2)), "float32")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        for n, oc_chunk, oh, ow, oc_block, ic, kh, kw in T.grid(T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2), T.int64(4), T.int64(2), T.int64(2)):
            with T.block("conv2d_NCHWc"):
                v_n, v_oc_chunk, v_oh, v_ow, v_oc_block, v_ic, v_kh, v_kw = T.axis.remap("SSSSSRRR", [n, oc_chunk, oh, ow, oc_block, ic, kh, kw])
                T.reads(A[v_n, v_ic // T.int64(2), v_oh + v_kh, v_ow + v_kw, v_ic % T.int64(2)]

In [94]:
relax_mod.show()

In [95]:
my_instrument = MyInstrument()
with tvm.transform.PassContext(instruments=[my_instrument]):
    ex = relax.build(relax_mod, target, exec_mode="compiled", pipeline="micro_build")

run_before_pass _pipeline
['__class__', '__contains__', '__del__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_handle_by_constructor__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__setstate__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_add', '_import', '_move', '_relax_script', 'astext', 'attrs', 'clone', 'from_expr', 'functions', 'functions_items', 'get_attr', 'get_constructor', 'get_global_type_var', 'get_global_type_vars', 'get_global_var', 'get_global_vars', 'get_type', 'global_infos', 'global_type_var_map_', 'global_var_map_', 'handle', 'import_from_std', 'legacy_repr', 'same_as', 'script', 'show', 'source_map', 'type_definitions', 'update', 'update_func', 'update_global_info', 'with

TVMError: Traceback (most recent call last):
  8: tvm::runtime::PackedFuncObj::Extractor<tvm::runtime::PackedFuncSubObj<tvm::runtime::TypedPackedFunc<tvm::IRModule (tvm::relax::ExecBuilder, tvm::IRModule)>::AssignTypedLambda<tvm::IRModule (*)(tvm::relax::ExecBuilder, tvm::IRModule)>(tvm::IRModule (*)(tvm::relax::ExecBuilder, tvm::IRModule), std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >)::{lambda(tvm::runtime::TVMArgs const&, tvm::runtime::TVMRetValue*)#1}> >::Call(tvm::runtime::PackedFuncObj const*, tvm::runtime::TVMArgs, tvm::runtime::TVMRetValue*)
  7: tvm::relax::relax_vm::VMTIRCodeGen(tvm::relax::ExecBuilder, tvm::IRModule)
  6: tvm::relax::relax_vm::CodeGenVMTIR::Run(tvm::relax::ExecBuilder, tvm::IRModule)
  5: tvm::relax::relax_vm::CodeGenVMTIR::Codegen(tvm::relax::Function const&)
  4: tvm::relax::ExprFunctor<tvm::runtime::Optional<tvm::PrimExpr> (tvm::RelayExpr const&)>::VisitExpr(tvm::RelayExpr const&)
  3: _ZZN3tvm5relax11ExprFunctorIFNS_7run

In [97]:
print(my_instrument.output[-1][0])

metadata = tvm.ir.load_json("""{
  \"root\": 1, 
  \"nodes\": [
    {
      \"type_key\": \"\"
    }, 
    {
      \"type_key\": \"Map\", 
      \"keys\": [
        \"relax.expr.Constant\"
      ], 
      \"data\": [2]
    }, 
    {
      \"type_key\": \"Array\", 
      \"data\": [3]
    }, 
    {
      \"type_key\": \"relax.expr.Constant\", 
      \"attrs\": {
        \"_checked_type_\": \"15\", 
        \"data\": \"0\", 
        \"span\": \"0\", 
        \"struct_info_\": \"4\"
      }
    }, 
    {
      \"type_key\": \"relax.TensorStructInfo\", 
      \"attrs\": {
        \"dtype\": \"float32\", 
        \"ndim\": \"6\", 
        \"shape\": \"5\", 
        \"span\": \"0\", 
        \"vdevice\": \"0\"
      }
    }, 
    {
      \"type_key\": \"relax.expr.ShapeExpr\", 
      \"attrs\": {
        \"_checked_type_\": \"14\", 
        \"span\": \"0\", 
        \"struct_info_\": \"13\", 
        \"values\": \"6\"
      }
    }, 
    {
      \"type_key\": \"Array\", 
      \"data\": [7, 

In [137]:
from tvm.script import tir as T
def foo():

    @T.prim_func
    def __vmtir__main(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""})})
        arg_type_ids_1 = T.decl_buffer((4,), "int32", data=arg_type_ids)
        ctx_ptr_code: T.int32 = arg_type_ids_1[0]
        r_code: T.int32 = arg_type_ids_1[1]
        c_code: T.int32 = arg_type_ids_1[2]
        f_code: T.int32 = arg_type_ids_1[3]
        ctx_ptr: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        r: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        c: T.handle = T.tvm_struct_get(args, 2, 12, "handle")
        f: T.handle = T.tvm_struct_get(args, 3, 12, "handle")
        with T.attr(0, "compute_scope", "__vmtir__main_compute_"):
            T.anylist_setitem_call_packed(r, 2, "vm.builtin.alloc_storage", ctx_ptr, T.anylist_getitem(c, 2), T.int64(0), T.anylist_getitem(c, 3), T.anylist_getitem(c, 4))
            T.anylist_setitem_call_packed(r, 3, "vm.builtin.alloc_tensor", T.anylist_getitem(r, 2), T.int64(0), T.anylist_getitem(c, 5), T.anylist_getitem(c, 6))
            T.anylist_setitem_call_packed(r, 2, "vm.builtin.null_value")
            T.call_cpacked("layout_transform", T.anylist_getitem(r, 0), T.anylist_getitem(r, 3), T.reinterpret("handle", T.uint64(0)))
            T.anylist_setitem_call_packed(r, 4, "vm.builtin.alloc_storage", ctx_ptr, T.anylist_getitem(c, 7), T.int64(0), T.anylist_getitem(c, 8), T.anylist_getitem(c, 4))
            T.anylist_setitem_call_packed(r, 5, "vm.builtin.alloc_tensor", T.anylist_getitem(r, 4), T.int64(0), T.anylist_getitem(c, 9), T.anylist_getitem(c, 10))
            T.anylist_setitem_call_packed(r, 4, "vm.builtin.null_value")
            T.call_cpacked("contrib_conv2d_NCHWc", T.anylist_getitem(r, 3), T.anylist_getitem(c, 11), T.anylist_getitem(r, 5), T.reinterpret("handle", T.uint64(0)))
            T.anylist_setitem_call_packed(r, 3, "vm.builtin.null_value")
            T.anylist_setitem_call_packed(r, 6, "vm.builtin.alloc_storage", ctx_ptr, T.anylist_getitem(c, 7), T.int64(0), T.anylist_getitem(c, 12), T.anylist_getitem(c, 4))
            T.anylist_setitem_call_packed(r, 7, "vm.builtin.alloc_tensor", T.anylist_getitem(r, 6), T.int64(0), T.anylist_getitem(c, 13), T.anylist_getitem(c, 14))
            T.anylist_setitem_call_packed(r, 6, "vm.builtin.null_value")
            T.call_cpacked("layout_transform1", T.anylist_getitem(r, 5), T.anylist_getitem(r, 7), T.reinterpret("handle", T.uint64(0)))
            T.anylist_setitem_call_packed(r, 5, "vm.builtin.null_value")
            T.anylist_setitem_call_packed(r, 1, "vm.builtin.copy", T.anylist_getitem(r, 7))
        return 0
    
    @T.prim_func
    def contrib_conv2d_NCHWc(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""}), "tir.noalias": T.bool(True)})
        arg_type_ids_1 = T.decl_buffer((3,), "int32", data=arg_type_ids)
        var_A_code: T.int32 = arg_type_ids_1[0]
        var_B_code: T.int32 = arg_type_ids_1[1]
        var_conv2d_NCHWc_code: T.int32 = arg_type_ids_1[2]
        var_A: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        var_B: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        var_conv2d_NCHWc: T.handle = T.tvm_struct_get(args, 2, 12, "handle")
        contrib_conv2d_NCHWc_var_A_shape: T.handle("int64") = T.tvm_struct_get(var_A, 0, 2, "handle")
        contrib_conv2d_NCHWc_var_A_shape_1 = T.decl_buffer((5,), "int64", data=contrib_conv2d_NCHWc_var_A_shape)
        contrib_conv2d_NCHWc_var_A_strides: T.handle("int64") = T.tvm_struct_get(var_A, 0, 3, "handle")
        contrib_conv2d_NCHWc_var_A_strides_1 = T.decl_buffer((0,), "int64", data=contrib_conv2d_NCHWc_var_A_strides)
        dev_id: T.int32 = T.tvm_struct_get(var_A, 0, 9, "int32")
        A: T.handle("float32", "global") = T.tvm_struct_get(var_A, 0, 1, "handle")
        T.attr(A, "storage_alignment", 64)
        contrib_conv2d_NCHWc_var_B_shape: T.handle("int64") = T.tvm_struct_get(var_B, 0, 2, "handle")
        contrib_conv2d_NCHWc_var_B_shape_1 = T.decl_buffer((6,), "int64", data=contrib_conv2d_NCHWc_var_B_shape)
        contrib_conv2d_NCHWc_var_B_strides: T.handle("int64") = T.tvm_struct_get(var_B, 0, 3, "handle")
        contrib_conv2d_NCHWc_var_B_strides_1 = T.decl_buffer((0,), "int64", data=contrib_conv2d_NCHWc_var_B_strides)
        B: T.handle("float32", "global") = T.tvm_struct_get(var_B, 0, 1, "handle")
        T.attr(B, "storage_alignment", 64)
        contrib_conv2d_NCHWc_var_conv2d_NCHWc_shape: T.handle("int64") = T.tvm_struct_get(var_conv2d_NCHWc, 0, 2, "handle")
        contrib_conv2d_NCHWc_var_conv2d_NCHWc_shape_1 = T.decl_buffer((5,), "int64", data=contrib_conv2d_NCHWc_var_conv2d_NCHWc_shape)
        contrib_conv2d_NCHWc_var_conv2d_NCHWc_strides: T.handle("int64") = T.tvm_struct_get(var_conv2d_NCHWc, 0, 3, "handle")
        contrib_conv2d_NCHWc_var_conv2d_NCHWc_strides_1 = T.decl_buffer((0,), "int64", data=contrib_conv2d_NCHWc_var_conv2d_NCHWc_strides)
        conv2d_NCHWc: T.handle("float32", "global") = T.tvm_struct_get(var_conv2d_NCHWc, 0, 1, "handle")
        T.attr(conv2d_NCHWc, "storage_alignment", 64)
        T.attr("default", "device_id", dev_id)
        T.attr("default", "device_type", 1)
        if not T.isnullptr(contrib_conv2d_NCHWc_var_A_strides):
            T.evaluate(0)
        if not T.isnullptr(contrib_conv2d_NCHWc_var_B_strides):
            T.evaluate(0)
        if not T.isnullptr(contrib_conv2d_NCHWc_var_conv2d_NCHWc_strides):
            T.evaluate(0)
        A_1 = T.decl_buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), data=A)
        B_1 = T.decl_buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), data=B)
        conv2d_NCHWc_1 = T.decl_buffer((T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2)), data=conv2d_NCHWc)
        with T.attr(0, "compute_scope", "contrib_conv2d_NCHWc_compute_"):
            for oc_block, ic, kh, kw in T.grid(2, 4, 2, 2):
                cse_var_2: T.int32 = ic // 2
                cse_var_1: T.int32 = ic % 2
                conv2d_NCHWc_2 = T.Buffer((T.int64(2),), data=conv2d_NCHWc)
                if ic == 0 and kh == 0 and kw == 0:
                    conv2d_NCHWc_2[oc_block] = T.float32(0)
                A_2 = T.Buffer((T.int64(16),), data=A)
                B_2 = T.Buffer((T.int64(32),), data=B)
                conv2d_NCHWc_2[oc_block] = conv2d_NCHWc_2[oc_block] + A_2[cse_var_2 * 8 + kh * 4 + kw * 2 + cse_var_1] * B_2[cse_var_2 * 16 + kh * 8 + kw * 4 + cse_var_1 * 2 + oc_block]
        return 0

    @T.prim_func
    def layout_transform(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""}), "tir.noalias": T.bool(True)})
        arg_type_ids_1 = T.decl_buffer((2,), "int32", data=arg_type_ids)
        var_A_code: T.int32 = arg_type_ids_1[0]
        var_T_layout_trans_code: T.int32 = arg_type_ids_1[1]
        var_A: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        var_T_layout_trans: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        layout_transform_var_A_shape: T.handle("int64") = T.tvm_struct_get(var_A, 0, 2, "handle")
        layout_transform_var_A_shape_1 = T.decl_buffer((4,), "int64", data=layout_transform_var_A_shape)
        layout_transform_var_A_strides: T.handle("int64") = T.tvm_struct_get(var_A, 0, 3, "handle")
        layout_transform_var_A_strides_1 = T.decl_buffer((0,), "int64", data=layout_transform_var_A_strides)
        dev_id: T.int32 = T.tvm_struct_get(var_A, 0, 9, "int32")
        A: T.handle("float32", "global") = T.tvm_struct_get(var_A, 0, 1, "handle")
        T.attr(A, "storage_alignment", 64)
        layout_transform_var_T_layout_trans_shape: T.handle("int64") = T.tvm_struct_get(var_T_layout_trans, 0, 2, "handle")
        layout_transform_var_T_layout_trans_shape_1 = T.decl_buffer((5,), "int64", data=layout_transform_var_T_layout_trans_shape)
        layout_transform_var_T_layout_trans_strides: T.handle("int64") = T.tvm_struct_get(var_T_layout_trans, 0, 3, "handle")
        layout_transform_var_T_layout_trans_strides_1 = T.decl_buffer((0,), "int64", data=layout_transform_var_T_layout_trans_strides)
        T_layout_trans: T.handle("float32", "global") = T.tvm_struct_get(var_T_layout_trans, 0, 1, "handle")
        T.attr(T_layout_trans, "storage_alignment", 64)
        T.attr("default", "device_id", dev_id)
        T.attr("default", "device_type", 1)
        if not T.isnullptr(layout_transform_var_A_strides):
            T.evaluate(0)
        if not T.isnullptr(layout_transform_var_T_layout_trans_strides):
            T.evaluate(0)
        A_1 = T.decl_buffer((T.int64(1), T.int64(4), T.int64(2), T.int64(2)), data=A)
        T_layout_trans_1 = T.decl_buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), data=T_layout_trans)
        with T.attr(0, "compute_scope", "layout_transform_compute_"):
            for ax1, ax2, ax3, ax4 in T.grid(2, 2, 2, 2):
                cse_var_1: T.int32 = ax1 * 8
                T_layout_trans_2 = T.Buffer((T.int64(16),), data=T_layout_trans)
                A_2 = T.Buffer((T.int64(16),), data=A)
                T_layout_trans_2[cse_var_1 + ax2 * 4 + ax3 * 2 + ax4] = A_2[cse_var_1 + ax4 * 4 + ax2 * 2 + ax3]
        return 0
        
    with tvm.transform.PassContext(config={"tir.disable_vectorize": True}):
        mod = tvm.build(__vmtir__main, target="llvm")
    return mod
mod = foo()

In [138]:
mod

Module(llvm, 18fff4d8)

In [139]:
print(mod.get_source())

; ModuleID = 'TVMMod'
source_filename = "TVMMod"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"

%0 = type { double }

@__tvm_module_ctx = linkonce dllexport local_unnamed_addr global ptr null, align 8
@__TVMFuncCall = linkonce dllexport local_unnamed_addr global ptr null, align 8
@__TVMBackendGetFuncFromEnv = linkonce dllexport local_unnamed_addr global ptr null, align 8
@.tvm_func.vm.builtin.alloc_storage = internal unnamed_addr global ptr null, align 8
@.str = private constant [25 x i8] c"vm.builtin.alloc_storage\00", align 1
@.tvm_func.vm.builtin.alloc_tensor = internal unnamed_addr global ptr null, align 8
@.str.1 = private constant [24 x i8] c"vm.builtin.alloc_tensor\00", align 1
@.tvm_func.vm.builtin.null_value = internal unnamed_addr global ptr null, align 8
@.str.2 = private constant [22 x i8] c"vm.builtin.null_value\00", align 1
@.tvm_func.vm.builtin.copy = internal unnamed_addr global pt

In [4]:
from tvm.script import ir as I
from tvm.script import tir as T

@I.ir_module
class Module:
    I.module_attrs({"runtime": None})
    @T.prim_func
    def __vmtir__main(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""})})
        arg_type_ids_1 = T.decl_buffer((4,), "int32", data=arg_type_ids)
        ctx_ptr_code: T.int32 = arg_type_ids_1[0]
        r_code: T.int32 = arg_type_ids_1[1]
        c_code: T.int32 = arg_type_ids_1[2]
        f_code: T.int32 = arg_type_ids_1[3]
        ctx_ptr: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        r: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        c: T.handle = T.tvm_struct_get(args, 2, 12, "handle")
        f: T.handle = T.tvm_struct_get(args, 3, 12, "handle")
        with T.attr(0, "compute_scope", "__vmtir__main_compute_"):
            T.anylist_setitem_call_packed(r, 2, "vm.builtin.alloc_storage", ctx_ptr, T.anylist_getitem(c, 2), T.int64(0), T.anylist_getitem(c, 3), T.anylist_getitem(c, 4))
            T.anylist_setitem_call_packed(r, 3, "vm.builtin.alloc_tensor", T.anylist_getitem(r, 2), T.int64(0), T.anylist_getitem(c, 5), T.anylist_getitem(c, 6))
            T.anylist_setitem_call_packed(r, 2, "vm.builtin.null_value")
            T.call_cpacked("layout_transform", T.anylist_getitem(r, 0), T.anylist_getitem(r, 3), T.reinterpret("handle", T.uint64(0)))
            T.anylist_setitem_call_packed(r, 4, "vm.builtin.alloc_storage", ctx_ptr, T.anylist_getitem(c, 7), T.int64(0), T.anylist_getitem(c, 8), T.anylist_getitem(c, 4))
            T.anylist_setitem_call_packed(r, 5, "vm.builtin.alloc_tensor", T.anylist_getitem(r, 4), T.int64(0), T.anylist_getitem(c, 9), T.anylist_getitem(c, 10))
            T.anylist_setitem_call_packed(r, 4, "vm.builtin.null_value")
            T.call_cpacked("contrib_conv2d_NCHWc", T.anylist_getitem(r, 3), T.anylist_getitem(c, 11), T.anylist_getitem(r, 5), T.reinterpret("handle", T.uint64(0)))
            T.anylist_setitem_call_packed(r, 3, "vm.builtin.null_value")
            T.anylist_setitem_call_packed(r, 6, "vm.builtin.alloc_storage", ctx_ptr, T.anylist_getitem(c, 7), T.int64(0), T.anylist_getitem(c, 12), T.anylist_getitem(c, 4))
            T.anylist_setitem_call_packed(r, 7, "vm.builtin.alloc_tensor", T.anylist_getitem(r, 6), T.int64(0), T.anylist_getitem(c, 13), T.anylist_getitem(c, 14))
            T.anylist_setitem_call_packed(r, 6, "vm.builtin.null_value")
            T.call_cpacked("layout_transform1", T.anylist_getitem(r, 5), T.anylist_getitem(r, 7), T.reinterpret("handle", T.uint64(0)))
            T.anylist_setitem_call_packed(r, 5, "vm.builtin.null_value")
            T.anylist_setitem_call_packed(r, 1, "vm.builtin.copy", T.anylist_getitem(r, 7))
        return 0

    @T.prim_func
    def contrib_conv2d_NCHWc(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""}), "tir.noalias": T.bool(True)})
        arg_type_ids_1 = T.decl_buffer((3,), "int32", data=arg_type_ids)
        var_A_code: T.int32 = arg_type_ids_1[0]
        var_B_code: T.int32 = arg_type_ids_1[1]
        var_conv2d_NCHWc_code: T.int32 = arg_type_ids_1[2]
        var_A: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        var_B: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        var_conv2d_NCHWc: T.handle = T.tvm_struct_get(args, 2, 12, "handle")
        contrib_conv2d_NCHWc_var_A_shape: T.handle("int64") = T.tvm_struct_get(var_A, 0, 2, "handle")
        contrib_conv2d_NCHWc_var_A_shape_1 = T.decl_buffer((5,), "int64", data=contrib_conv2d_NCHWc_var_A_shape)
        contrib_conv2d_NCHWc_var_A_strides: T.handle("int64") = T.tvm_struct_get(var_A, 0, 3, "handle")
        contrib_conv2d_NCHWc_var_A_strides_1 = T.decl_buffer((0,), "int64", data=contrib_conv2d_NCHWc_var_A_strides)
        dev_id: T.int32 = T.tvm_struct_get(var_A, 0, 9, "int32")
        A: T.handle("float32", "global") = T.tvm_struct_get(var_A, 0, 1, "handle")
        T.attr(A, "storage_alignment", 64)
        contrib_conv2d_NCHWc_var_B_shape: T.handle("int64") = T.tvm_struct_get(var_B, 0, 2, "handle")
        contrib_conv2d_NCHWc_var_B_shape_1 = T.decl_buffer((6,), "int64", data=contrib_conv2d_NCHWc_var_B_shape)
        contrib_conv2d_NCHWc_var_B_strides: T.handle("int64") = T.tvm_struct_get(var_B, 0, 3, "handle")
        contrib_conv2d_NCHWc_var_B_strides_1 = T.decl_buffer((0,), "int64", data=contrib_conv2d_NCHWc_var_B_strides)
        B: T.handle("float32", "global") = T.tvm_struct_get(var_B, 0, 1, "handle")
        T.attr(B, "storage_alignment", 64)
        contrib_conv2d_NCHWc_var_conv2d_NCHWc_shape: T.handle("int64") = T.tvm_struct_get(var_conv2d_NCHWc, 0, 2, "handle")
        contrib_conv2d_NCHWc_var_conv2d_NCHWc_shape_1 = T.decl_buffer((5,), "int64", data=contrib_conv2d_NCHWc_var_conv2d_NCHWc_shape)
        contrib_conv2d_NCHWc_var_conv2d_NCHWc_strides: T.handle("int64") = T.tvm_struct_get(var_conv2d_NCHWc, 0, 3, "handle")
        contrib_conv2d_NCHWc_var_conv2d_NCHWc_strides_1 = T.decl_buffer((0,), "int64", data=contrib_conv2d_NCHWc_var_conv2d_NCHWc_strides)
        conv2d_NCHWc: T.handle("float32", "global") = T.tvm_struct_get(var_conv2d_NCHWc, 0, 1, "handle")
        T.attr(conv2d_NCHWc, "storage_alignment", 64)
        T.attr("default", "device_id", dev_id)
        T.attr("default", "device_type", 1)
        if not T.isnullptr(contrib_conv2d_NCHWc_var_A_strides):
            T.evaluate(0)
        if not T.isnullptr(contrib_conv2d_NCHWc_var_B_strides):
            T.evaluate(0)
        if not T.isnullptr(contrib_conv2d_NCHWc_var_conv2d_NCHWc_strides):
            T.evaluate(0)
        A_1 = T.decl_buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), data=A)
        B_1 = T.decl_buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), data=B)
        conv2d_NCHWc_1 = T.decl_buffer((T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2)), data=conv2d_NCHWc)
        with T.attr(0, "compute_scope", "contrib_conv2d_NCHWc_compute_"):
            for oc_block, ic, kh, kw in T.grid(2, 4, 2, 2):
                cse_var_2: T.int32 = ic // 2
                cse_var_1: T.int32 = ic % 2
                conv2d_NCHWc_2 = T.Buffer((T.int64(2),), data=conv2d_NCHWc)
                if ic == 0 and kh == 0 and kw == 0:
                    conv2d_NCHWc_2[oc_block] = T.float32(0)
                A_2 = T.Buffer((T.int64(16),), data=A)
                B_2 = T.Buffer((T.int64(32),), data=B)
                conv2d_NCHWc_2[oc_block] = conv2d_NCHWc_2[oc_block] + A_2[cse_var_2 * 8 + kh * 4 + kw * 2 + cse_var_1] * B_2[cse_var_2 * 16 + kh * 8 + kw * 4 + cse_var_1 * 2 + oc_block]
        return 0

    @T.prim_func
    def layout_transform(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""}), "tir.noalias": T.bool(True)})
        arg_type_ids_1 = T.decl_buffer((2,), "int32", data=arg_type_ids)
        var_A_code: T.int32 = arg_type_ids_1[0]
        var_T_layout_trans_code: T.int32 = arg_type_ids_1[1]
        var_A: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        var_T_layout_trans: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        layout_transform_var_A_shape: T.handle("int64") = T.tvm_struct_get(var_A, 0, 2, "handle")
        layout_transform_var_A_shape_1 = T.decl_buffer((4,), "int64", data=layout_transform_var_A_shape)
        layout_transform_var_A_strides: T.handle("int64") = T.tvm_struct_get(var_A, 0, 3, "handle")
        layout_transform_var_A_strides_1 = T.decl_buffer((0,), "int64", data=layout_transform_var_A_strides)
        dev_id: T.int32 = T.tvm_struct_get(var_A, 0, 9, "int32")
        A: T.handle("float32", "global") = T.tvm_struct_get(var_A, 0, 1, "handle")
        T.attr(A, "storage_alignment", 64)
        layout_transform_var_T_layout_trans_shape: T.handle("int64") = T.tvm_struct_get(var_T_layout_trans, 0, 2, "handle")
        layout_transform_var_T_layout_trans_shape_1 = T.decl_buffer((5,), "int64", data=layout_transform_var_T_layout_trans_shape)
        layout_transform_var_T_layout_trans_strides: T.handle("int64") = T.tvm_struct_get(var_T_layout_trans, 0, 3, "handle")
        layout_transform_var_T_layout_trans_strides_1 = T.decl_buffer((0,), "int64", data=layout_transform_var_T_layout_trans_strides)
        T_layout_trans: T.handle("float32", "global") = T.tvm_struct_get(var_T_layout_trans, 0, 1, "handle")
        T.attr(T_layout_trans, "storage_alignment", 64)
        T.attr("default", "device_id", dev_id)
        T.attr("default", "device_type", 1)
        if not T.isnullptr(layout_transform_var_A_strides):
            T.evaluate(0)
        if not T.isnullptr(layout_transform_var_T_layout_trans_strides):
            T.evaluate(0)
        A_1 = T.decl_buffer((T.int64(1), T.int64(4), T.int64(2), T.int64(2)), data=A)
        T_layout_trans_1 = T.decl_buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), data=T_layout_trans)
        with T.attr(0, "compute_scope", "layout_transform_compute_"):
            for ax1, ax2, ax3, ax4 in T.grid(2, 2, 2, 2):
                cse_var_1: T.int32 = ax1 * 8
                T_layout_trans_2 = T.Buffer((T.int64(16),), data=T_layout_trans)
                A_2 = T.Buffer((T.int64(16),), data=A)
                T_layout_trans_2[cse_var_1 + ax2 * 4 + ax3 * 2 + ax4] = A_2[cse_var_1 + ax4 * 4 + ax2 * 2 + ax3]
        return 0

    @T.prim_func
    def layout_transform1(args: T.handle, arg_type_ids: T.handle("int32"), num_args: T.int32, out_ret_value: T.handle("void"), out_ret_tcode: T.handle("int32"), resource_handle: T.handle) -> T.int32:
        T.func_attr({"calling_conv": 1, "target": T.target({"keys": ["cpu"], "kind": "llvm", "mtriple": "x86_64-unknown-linux-gnu", "tag": ""}), "tir.noalias": T.bool(True)})
        arg_type_ids_1 = T.decl_buffer((2,), "int32", data=arg_type_ids)
        var_A_code: T.int32 = arg_type_ids_1[0]
        var_T_layout_trans_code: T.int32 = arg_type_ids_1[1]
        var_A: T.handle = T.tvm_struct_get(args, 0, 12, "handle")
        var_T_layout_trans: T.handle = T.tvm_struct_get(args, 1, 12, "handle")
        layout_transform1_var_A_shape: T.handle("int64") = T.tvm_struct_get(var_A, 0, 2, "handle")
        layout_transform1_var_A_shape_1 = T.decl_buffer((5,), "int64", data=layout_transform1_var_A_shape)
        layout_transform1_var_A_strides: T.handle("int64") = T.tvm_struct_get(var_A, 0, 3, "handle")
        layout_transform1_var_A_strides_1 = T.decl_buffer((0,), "int64", data=layout_transform1_var_A_strides)
        dev_id: T.int32 = T.tvm_struct_get(var_A, 0, 9, "int32")
        A: T.handle("float32", "global") = T.tvm_struct_get(var_A, 0, 1, "handle")
        T.attr(A, "storage_alignment", 64)
        layout_transform1_var_T_layout_trans_shape: T.handle("int64") = T.tvm_struct_get(var_T_layout_trans, 0, 2, "handle")
        layout_transform1_var_T_layout_trans_shape_1 = T.decl_buffer((4,), "int64", data=layout_transform1_var_T_layout_trans_shape)
        layout_transform1_var_T_layout_trans_strides: T.handle("int64") = T.tvm_struct_get(var_T_layout_trans, 0, 3, "handle")
        layout_transform1_var_T_layout_trans_strides_1 = T.decl_buffer((0,), "int64", data=layout_transform1_var_T_layout_trans_strides)
        T_layout_trans: T.handle("float32", "global") = T.tvm_struct_get(var_T_layout_trans, 0, 1, "handle")
        T.attr(T_layout_trans, "storage_alignment", 64)
        T.attr("default", "device_id", dev_id)
        T.attr("default", "device_type", 1)
        if not T.isnullptr(layout_transform1_var_A_strides):
            T.evaluate(0)
        if not T.isnullptr(layout_transform1_var_T_layout_trans_strides):
            T.evaluate(0)
        A_1 = T.decl_buffer((T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2)), data=A)
        T_layout_trans_1 = T.decl_buffer((T.int64(1), T.int64(2), T.int64(1), T.int64(1)), data=T_layout_trans)
        with T.attr(0, "compute_scope", "layout_transform1_compute_"):
            for ax1 in range(2):
                T_layout_trans_2 = T.Buffer((T.int64(2),), data=T_layout_trans)
                A_2 = T.Buffer((T.int64(2),), data=A)
                T_layout_trans_2[ax1] = A_2[ax1]
        return 0

In [5]:
with tvm.transform.PassContext(config={"tir.disable_vectorize": True}):
    mod = tvm.build(Module, target="llvm")
mod

Module(llvm, 15726268)

In [6]:
print(mod.get_source())

; ModuleID = 'TVMMod'
source_filename = "TVMMod"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"

%0 = type { double }

@__tvm_module_ctx = linkonce dllexport local_unnamed_addr global ptr null, align 8
@__TVMFuncCall = linkonce dllexport local_unnamed_addr global ptr null, align 8
@__TVMBackendGetFuncFromEnv = linkonce dllexport local_unnamed_addr global ptr null, align 8
@.tvm_func.vm.builtin.alloc_storage = internal unnamed_addr global ptr null, align 8
@.str = private constant [25 x i8] c"vm.builtin.alloc_storage\00", align 1
@.tvm_func.vm.builtin.alloc_tensor = internal unnamed_addr global ptr null, align 8
@.str.1 = private constant [24 x i8] c"vm.builtin.alloc_tensor\00", align 1
@.tvm_func.vm.builtin.null_value = internal unnamed_addr global ptr null, align 8
@.str.2 = private constant [22 x i8] c"vm.builtin.null_value\00", align 1
@.tvm_func.vm.builtin.copy = internal unnamed_addr global pt

In [ ]:
mod["__vmtir__main"]()

In [32]:
from tvm.script import ir as I
from tvm.script import tir as T
from tvm.script import relax as R

@I.ir_module
class Module:
    @T.prim_func(private=True)
    def contrib_conv2d_NCHWc(A: T.Buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), "float32"), B: T.Buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), "float32"), conv2d_NCHWc: T.Buffer((T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2)), "float32")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        for n, oc_chunk, oh, ow, oc_block, ic, kh, kw in T.grid(T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2), T.int64(4), T.int64(2), T.int64(2)):
            with T.block("conv2d_NCHWc"):
                v_n, v_oc_chunk, v_oh, v_ow, v_oc_block, v_ic, v_kh, v_kw = T.axis.remap("SSSSSRRR", [n, oc_chunk, oh, ow, oc_block, ic, kh, kw])
                T.reads(A[v_n, v_ic // T.int64(2), v_oh + v_kh, v_ow + v_kw, v_ic % T.int64(2)], B[v_oc_chunk, v_ic // T.int64(2), v_kh, v_kw, v_ic % T.int64(2), v_oc_block])
                T.writes(conv2d_NCHWc[v_n, v_oc_chunk, v_oh, v_ow, v_oc_block])
                with T.init():
                    conv2d_NCHWc[v_n, v_oc_chunk, v_oh, v_ow, v_oc_block] = T.float32(0)
                conv2d_NCHWc[v_n, v_oc_chunk, v_oh, v_ow, v_oc_block] = conv2d_NCHWc[v_n, v_oc_chunk, v_oh, v_ow, v_oc_block] + A[v_n, v_ic // T.int64(2), v_oh + v_kh, v_ow + v_kw, v_ic % T.int64(2)] * B[v_oc_chunk, v_ic // T.int64(2), v_kh, v_kw, v_ic % T.int64(2), v_oc_block]

    @T.prim_func(private=True)
    def layout_transform(A: T.Buffer((T.int64(1), T.int64(4), T.int64(2), T.int64(2)), "float32"), T_layout_trans: T.Buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), "float32")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        for ax0, ax1, ax2, ax3, ax4 in T.grid(T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)):
            with T.block("T_layout_trans"):
                v_ax0, v_ax1, v_ax2, v_ax3, v_ax4 = T.axis.remap("SSSSS", [ax0, ax1, ax2, ax3, ax4])
                T.reads(A[v_ax0, v_ax1 * T.int64(2) + v_ax4, v_ax2, v_ax3])
                T.writes(T_layout_trans[v_ax0, v_ax1, v_ax2, v_ax3, v_ax4])
                T.block_attr({"dst_layout": "NCHW2c", "input_shape": [T.int64(1), T.int64(4), T.int64(2), T.int64(2)], "schedule_rule": "None", "src_layout": "NCHW"})
                T_layout_trans[v_ax0, v_ax1, v_ax2, v_ax3, v_ax4] = T.if_then_else(v_ax0 < T.int64(1) and v_ax1 * T.int64(2) + v_ax4 < T.int64(4) and v_ax2 < T.int64(2) and v_ax3 < T.int64(2), A[v_ax0, v_ax1 * T.int64(2) + v_ax4, v_ax2, v_ax3], T.float32(0))

    @T.prim_func(private=True)
    def layout_transform1(A: T.Buffer((T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2)), "float32"), T_layout_trans: T.Buffer((T.int64(1), T.int64(2), T.int64(1), T.int64(1)), "float32")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        for ax0, ax1, ax2, ax3 in T.grid(T.int64(1), T.int64(2), T.int64(1), T.int64(1)):
            with T.block("T_layout_trans"):
                v_ax0, v_ax1, v_ax2, v_ax3 = T.axis.remap("SSSS", [ax0, ax1, ax2, ax3])
                T.reads(A[v_ax0, v_ax1 // T.int64(2), v_ax2, v_ax3, v_ax1 % T.int64(2)])
                T.writes(T_layout_trans[v_ax0, v_ax1, v_ax2, v_ax3])
                T.block_attr({"dst_layout": "NCHW", "input_shape": [T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2)], "schedule_rule": "None", "src_layout": "NCHW2c"})
                T_layout_trans[v_ax0, v_ax1, v_ax2, v_ax3] = T.if_then_else(v_ax0 < T.int64(1) and v_ax1 < T.int64(2) and v_ax2 < T.int64(1) and v_ax3 < T.int64(1), A[v_ax0, v_ax1 // T.int64(2), v_ax2, v_ax3, v_ax1 % T.int64(2)], T.float32(0))

    @R.function
    def main(x: R.Tensor((1, 4, 2, 2), dtype="float32")) -> R.Tensor((1, 2, 1, 1), dtype="float32"):
        R.func_attr({"relax.force_pure": 1})
        cls = Module
        lv = R.call_tir(cls.layout_transform, (x,), out_sinfo=R.Tensor((1, 2, 2, 2, 2), dtype="float32"))
        lv1 = R.call_tir(cls.contrib_conv2d_NCHWc, (lv, metadata["relax.expr.Constant"][0]), out_sinfo=R.Tensor((1, 1, 1, 1, 2), dtype="float32"))
        lv2 = R.call_tir(cls.layout_transform1, (lv1,), out_sinfo=R.Tensor((1, 2, 1, 1), dtype="float32"))
        gv: R.Tensor((1, 2, 1, 1), dtype="float32") = lv2
        return gv

error: Undefined variable: metadata
 --> /tmp/ipykernel_1993531/213334641.py:49:9
    |  
 49 |          lv1 = R.call_tir(cls.contrib_conv2d_NCHWc, (lv, metadata["relax.expr.Constant"][0]), out_sinfo=R.Tensor((1, 1, 1, 1, 2), dtype="float32"))
    |          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


DiagnosticError: Traceback (most recent call last):
  1: tvm::runtime::PackedFuncObj::Extractor<tvm::runtime::PackedFuncSubObj<tvm::runtime::TypedPackedFunc<void (tvm::DiagnosticContext)>::AssignTypedLambda<tvm::{lambda(tvm::DiagnosticContext)#9}>(tvm::{lambda(tvm::DiagnosticContext)#9}, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >)::{lambda(tvm::runtime::TVMArgs const&, tvm::runtime::TVMRetValue*)#1}> >::Call(tvm::runtime::PackedFuncObj const*, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >, tvm::runtime::TVMRetValue)
  0: tvm::DiagnosticContext::Render()
  File "/var/tmp/ga87puy/tvm_relax/src/ir/diagnostic.cc", line 131
DiagnosticError: one or more error diagnostics were emitted, please check diagnostic render for output.

In [98]:
metadata = tvm.ir.load_json("""{
  \"root\": 1, 
  \"nodes\": [
    {
      \"type_key\": \"\"
    }, 
    {
      \"type_key\": \"Map\", 
      \"keys\": [
        \"relax.expr.Constant\"
      ], 
      \"data\": [2]
    }, 
    {
      \"type_key\": \"Array\", 
      \"data\": [3]
    }, 
    {
      \"type_key\": \"relax.expr.Constant\", 
      \"attrs\": {
        \"_checked_type_\": \"15\", 
        \"data\": \"0\", 
        \"span\": \"0\", 
        \"struct_info_\": \"4\"
      }
    }, 
    {
      \"type_key\": \"relax.TensorStructInfo\", 
      \"attrs\": {
        \"dtype\": \"float32\", 
        \"ndim\": \"6\", 
        \"shape\": \"5\", 
        \"span\": \"0\", 
        \"vdevice\": \"0\"
      }
    }, 
    {
      \"type_key\": \"relax.expr.ShapeExpr\", 
      \"attrs\": {
        \"_checked_type_\": \"14\", 
        \"span\": \"0\", 
        \"struct_info_\": \"13\", 
        \"values\": \"6\"
      }
    }, 
    {
      \"type_key\": \"Array\", 
      \"data\": [7, 8, 9, 10, 11, 12]
    }, 
    {
      \"type_key\": \"IntImm\", 
      \"attrs\": {
        \"dtype\": \"int64\", 
        \"span\": \"0\", 
        \"value\": \"1\"
      }
    }, 
    {
      \"type_key\": \"IntImm\", 
      \"attrs\": {
        \"dtype\": \"int64\", 
        \"span\": \"0\", 
        \"value\": \"2\"
      }
    }, 
    {
      \"type_key\": \"IntImm\", 
      \"attrs\": {
        \"dtype\": \"int64\", 
        \"span\": \"0\", 
        \"value\": \"2\"
      }
    }, 
    {
      \"type_key\": \"IntImm\", 
      \"attrs\": {
        \"dtype\": \"int64\", 
        \"span\": \"0\", 
        \"value\": \"2\"
      }
    }, 
    {
      \"type_key\": \"IntImm\", 
      \"attrs\": {
        \"dtype\": \"int64\", 
        \"span\": \"0\", 
        \"value\": \"2\"
      }
    }, 
    {
      \"type_key\": \"IntImm\", 
      \"attrs\": {
        \"dtype\": \"int64\", 
        \"span\": \"0\", 
        \"value\": \"2\"
      }
    }, 
    {
      \"type_key\": \"relax.ShapeStructInfo\", 
      \"attrs\": {
        \"ndim\": \"6\", 
        \"span\": \"0\", 
        \"values\": \"6\"
      }
    }, 
    {
      \"type_key\": \"relax.ShapeType\", 
      \"attrs\": {
        \"ndim\": \"6\", 
        \"span\": \"0\"
      }
    }, 
    {
      \"type_key\": \"relax.DynTensorType\", 
      \"attrs\": {
        \"dtype\": \"float32\", 
        \"ndim\": \"6\", 
        \"span\": \"0\"
      }
    }
  ], 
  \"b64ndarrays\": [
    \"P6G0lvBAXt0AAAAAAAAAAAEAAAAAAAAABgAAAAIgAQABAAAAAAAAAAIAAAAAAAAAAgAAAAAAAAACAAAAAAAAAAIAAAAAAAAAAgAAAAAAAACAAAAAAAAAAMZAeT4YXOQ9YDSCPuBHNz+xgXQ/4OmVPlxsvT7ehYc+VlMQP3hzmz46emY/F8rZPoAQCz95Ci0/4om/Pax95z12miQ/mWg2P8T0ND/wU/o+C1fKPhX5rz6UMV0/c1t3P4kz/D14HXA+fT8EPwrL8T7BKxI/uAT6PgR9MT9aQ9o+\"
  ], 
  \"attrs\": {\"tvm_version\": \"0.17.dev0\"}
}""")
# from tvm.script import ir as I
# from tvm.script import tir as T
# from tvm.script import relax as R

@I.ir_module
class Module:
    @T.prim_func(private=True)
    def contrib_conv2d_NCHWc(A: T.Buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), "float32"), B: T.Buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), "float32"), conv2d_NCHWc: T.Buffer((T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2)), "float32")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        for n, oc_chunk, oh, ow, oc_block, ic, kh, kw in T.grid(T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2), T.int64(4), T.int64(2), T.int64(2)):
            with T.block("conv2d_NCHWc"):
                v_n, v_oc_chunk, v_oh, v_ow, v_oc_block, v_ic, v_kh, v_kw = T.axis.remap("SSSSSRRR", [n, oc_chunk, oh, ow, oc_block, ic, kh, kw])
                T.reads(A[v_n, v_ic // T.int64(2), v_oh + v_kh, v_ow + v_kw, v_ic % T.int64(2)], B[v_oc_chunk, v_ic // T.int64(2), v_kh, v_kw, v_ic % T.int64(2), v_oc_block])
                T.writes(conv2d_NCHWc[v_n, v_oc_chunk, v_oh, v_ow, v_oc_block])
                with T.init():
                    conv2d_NCHWc[v_n, v_oc_chunk, v_oh, v_ow, v_oc_block] = T.float32(0)
                conv2d_NCHWc[v_n, v_oc_chunk, v_oh, v_ow, v_oc_block] = conv2d_NCHWc[v_n, v_oc_chunk, v_oh, v_ow, v_oc_block] + A[v_n, v_ic // T.int64(2), v_oh + v_kh, v_ow + v_kw, v_ic % T.int64(2)] * B[v_oc_chunk, v_ic // T.int64(2), v_kh, v_kw, v_ic % T.int64(2), v_oc_block]

    @T.prim_func(private=True)
    def layout_transform(A: T.Buffer((T.int64(1), T.int64(4), T.int64(2), T.int64(2)), "float32"), T_layout_trans: T.Buffer((T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)), "float32")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        for ax0, ax1, ax2, ax3, ax4 in T.grid(T.int64(1), T.int64(2), T.int64(2), T.int64(2), T.int64(2)):
            with T.block("T_layout_trans"):
                v_ax0, v_ax1, v_ax2, v_ax3, v_ax4 = T.axis.remap("SSSSS", [ax0, ax1, ax2, ax3, ax4])
                T.reads(A[v_ax0, v_ax1 * T.int64(2) + v_ax4, v_ax2, v_ax3])
                T.writes(T_layout_trans[v_ax0, v_ax1, v_ax2, v_ax3, v_ax4])
                T.block_attr({"dst_layout": "NCHW2c", "input_shape": [T.int64(1), T.int64(4), T.int64(2), T.int64(2)], "schedule_rule": "None", "src_layout": "NCHW"})
                T_layout_trans[v_ax0, v_ax1, v_ax2, v_ax3, v_ax4] = T.if_then_else(v_ax0 < T.int64(1) and v_ax1 * T.int64(2) + v_ax4 < T.int64(4) and v_ax2 < T.int64(2) and v_ax3 < T.int64(2), A[v_ax0, v_ax1 * T.int64(2) + v_ax4, v_ax2, v_ax3], T.float32(0))

    @T.prim_func(private=True)
    def layout_transform1(A: T.Buffer((T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2)), "float32"), T_layout_trans: T.Buffer((T.int64(1), T.int64(2), T.int64(1), T.int64(1)), "float32")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        for ax0, ax1, ax2, ax3 in T.grid(T.int64(1), T.int64(2), T.int64(1), T.int64(1)):
            with T.block("T_layout_trans"):
                v_ax0, v_ax1, v_ax2, v_ax3 = T.axis.remap("SSSS", [ax0, ax1, ax2, ax3])
                T.reads(A[v_ax0, v_ax1 // T.int64(2), v_ax2, v_ax3, v_ax1 % T.int64(2)])
                T.writes(T_layout_trans[v_ax0, v_ax1, v_ax2, v_ax3])
                T.block_attr({"dst_layout": "NCHW", "input_shape": [T.int64(1), T.int64(1), T.int64(1), T.int64(1), T.int64(2)], "schedule_rule": "None", "src_layout": "NCHW2c"})
                T_layout_trans[v_ax0, v_ax1, v_ax2, v_ax3] = T.if_then_else(v_ax0 < T.int64(1) and v_ax1 < T.int64(2) and v_ax2 < T.int64(1) and v_ax3 < T.int64(1), A[v_ax0, v_ax1 // T.int64(2), v_ax2, v_ax3, v_ax1 % T.int64(2)], T.float32(0))

    @R.function
    def main(x: R.Tensor((1, 4, 2, 2), dtype="float32")) -> R.Tensor((1, 2, 1, 1), dtype="float32"):
        R.func_attr({"relax.force_pure": 1})
        cls = Module
        storage: R.Object = R.memory.alloc_storage(R.shape([64]), R.prim_value(0), R.str("global"), R.dtype("float32"))
        alloc: R.Tensor((1, 2, 2, 2, 2), dtype="float32") = R.memory.alloc_tensor(storage, R.prim_value(0), R.shape([1, 2, 2, 2, 2]), R.dtype("float32"))
        R.memory.kill_storage(storage)
        cls.layout_transform(x, alloc)
        storage1: R.Object = R.memory.alloc_storage(R.shape([8]), R.prim_value(0), R.str("global"), R.dtype("float32"))
        alloc1: R.Tensor((1, 1, 1, 1, 2), dtype="float32") = R.memory.alloc_tensor(storage1, R.prim_value(0), R.shape([1, 1, 1, 1, 2]), R.dtype("float32"))
        R.memory.kill_storage(storage1)
        cls.contrib_conv2d_NCHWc(alloc, metadata["relax.expr.Constant"][0], alloc1)
        R.memory.kill_tensor(alloc)
        storage_1: R.Object = R.memory.alloc_storage(R.shape([8]), R.prim_value(0), R.str("global"), R.dtype("uint8"))
        alloc2: R.Tensor((1, 2, 1, 1), dtype="float32") = R.memory.alloc_tensor(storage_1, R.prim_value(0), R.shape([1, 2, 1, 1]), R.dtype("float32"))
        R.memory.kill_storage(storage_1)
        cls.layout_transform1(alloc1, alloc2)
        R.memory.kill_tensor(alloc1)
        return alloc2



In [104]:
executor = relay.backend.Executor("graph")
with tvm.transform.PassContext(config={"tir.disable_vectorize": True}):
    mod = tvm.build(Module, target="llvm")
mod

InternalError: Traceback (most recent call last):
  10: tvm::runtime::PackedFuncObj::Extractor<tvm::runtime::PackedFuncSubObj<tvm::runtime::TypedPackedFunc<tvm::runtime::Module (tvm::runtime::Map<tvm::Target, tvm::IRModule, void, void> const&, tvm::Target)>::AssignTypedLambda<tvm::{lambda(tvm::runtime::Map<tvm::Target, tvm::IRModule, void, void> const&, tvm::Target)#6}>(tvm::{lambda(tvm::runtime::Map<tvm::Target, tvm::IRModule, void, void> const&, tvm::Target)#6}, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >)::{lambda(tvm::runtime::TVMArgs const&, tvm::runtime::TVMRetValue*)#1}> >::Call(tvm::runtime::PackedFuncObj const*, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >, tvm::runtime::TVMRetValue)
  9: tvm::TIRToRuntime(tvm::runtime::Map<tvm::Target, tvm::IRModule, void, void> const&, tvm::Target const&)
  8: tvm::SplitMixedModule(tvm::IRModule, tvm::Target const&, tvm::Target const&)
  7: tvm::ApplyPasses(tvm::IRModule,